# UEFA Euro 2024 Final — Veri Boru Hattı
**İspanya vs İngiltere** | Veri Çekimi · Önişleme · Kayıt

Bu notebook veriyi StatsBomb'dan çeker, temizler ve `cache/` klasörüne kaydeder.
Görselleştirmeler için `02_visualizations.ipynb` dosyasını çalıştırın.

## 0. Kurulum

In [1]:
!pip install statsbombpy pandas numpy -q

## 1. Kütüphaneler

In [2]:
import os, io, time, warnings, json
import numpy as np
import pandas as pd
from statsbombpy import sb
warnings.filterwarnings('ignore')

CACHE_DIR = r'D:\Masaüstü\Projects\MatchAnalysis\Spain-England2024\cache'
os.makedirs(CACHE_DIR, exist_ok=True)
print(f'Cache klasörü: {CACHE_DIR}')

Cache klasörü: D:\Masaüstü\Projects\MatchAnalysis\Spain-England2024\cache


## 2. Veri Çekimi

In [3]:
def _fetch_with_retry(fn, *args, **kwargs):
    """Rate-limit hatalarında otomatik yeniden dener."""
    for attempt in range(6):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            wait = 15 * (attempt + 1)
            print(f'  Hata: {e}  |  {wait}s bekleniyor... ({attempt+1}/6)')
            time.sleep(wait)
    raise RuntimeError('API 6 denemede de erişemedi.')

# Yarışmalar
cache_comps = os.path.join(CACHE_DIR, 'competitions.json')
if os.path.exists(cache_comps):
    comps = pd.read_json(cache_comps)
    print('[CACHE] competitions yüklendi.')
else:
    comps = _fetch_with_retry(sb.competitions)
    comps.to_json(cache_comps)
    print('[API]   competitions çekildi ve kaydedildi.')

euro = comps[comps['competition_name'] == 'UEFA Euro']
print(euro[['competition_id', 'season_id', 'season_name']].to_string())

[API]   competitions çekildi ve kaydedildi.
    competition_id  season_id season_name
73              55        282        2024
74              55         43        2020


In [4]:
# UEFA Euro 2024
row_2024 = euro[euro['season_name'] == '2024'].iloc[0]
competition_id = int(row_2024['competition_id'])  # 55
season_id      = int(row_2024['season_id'])        # 282
print(f'Competition ID: {competition_id}  |  Season ID: {season_id}')

# Maçlar
cache_matches = os.path.join(CACHE_DIR, f'matches_{competition_id}_{season_id}.json')
if os.path.exists(cache_matches):
    matches = pd.read_json(cache_matches)
    print('[CACHE] matches yüklendi.')
else:
    matches = _fetch_with_retry(sb.matches, competition_id=competition_id, season_id=season_id)
    matches.to_json(cache_matches)
    print('[API]   matches çekildi ve kaydedildi.')

print(f'Toplam maç sayısı: {len(matches)}')

Competition ID: 55  |  Season ID: 282
[CACHE] matches yüklendi.
Toplam maç sayısı: 51


In [5]:
# İspanya - İngiltere finalini bul
mask = (
    matches['home_team'].str.contains('Spain', case=False, na=False) |
    matches['away_team'].str.contains('Spain', case=False, na=False)
) & (
    matches['home_team'].str.contains('England', case=False, na=False) |
    matches['away_team'].str.contains('England', case=False, na=False)
)
final_row = matches[mask].iloc[0]
match_id  = int(final_row['match_id'])

print(f'Maç ID  : {match_id}')
print(f'Sonuç   : {final_row["home_team"]}  {final_row["home_score"]} - {final_row["away_score"]}  {final_row["away_team"]}')
print(f'Tarih   : {final_row["match_date"]}')
print(f'Aşama   : {final_row.get("competition_stage", "?")}')

Maç ID  : 3943043
Sonuç   : Spain  2 - 1  England
Tarih   : 2024-07-14
Aşama   : Final


In [6]:
# Eventleri çek
cache_events = os.path.join(CACHE_DIR, f'events_{match_id}.json')
if os.path.exists(cache_events):
    events_raw = pd.read_json(cache_events)
    print('[CACHE] events yüklendi.')
else:
    events_raw = _fetch_with_retry(sb.events, match_id=match_id)
    events_raw.to_json(cache_events)
    print('[API]   events çekildi ve kaydedildi.')

print(f'Toplam event : {len(events_raw)}')
print(f'Sütun sayısı : {len(events_raw.columns)}')
print(events_raw['type'].value_counts().to_string())

[CACHE] events yüklendi.
Toplam event : 3304
Sütun sayısı : 92
type
Pass                 915
Ball Receipt*        876
Carry                757
Pressure             327
Ball Recovery         69
Duel                  68
Clearance             42
Block                 41
Goal Keeper           30
Dribble               25
Shot                  25
Dispossessed          21
Foul Committed        19
Foul Won              19
Miscontrol            15
Dribbled Past         11
50/50                  8
Substitution           7
Injury Stoppage        6
Interception           5
Half Start             4
Half End               4
Tactical Shift         3
Shield                 2
Starting XI            2
Referee Ball-Drop      2
Error                  1


## 3. Veri Ön İşleme

In [7]:
events = events_raw.copy()

# location -> x, y
events['x'] = events['location'].apply(lambda v: v[0] if isinstance(v, list) else np.nan)
events['y'] = events['location'].apply(lambda v: v[1] if isinstance(v, list) else np.nan)

# pas bitiş konumu
if 'pass_end_location' in events.columns:
    events['pass_end_x'] = events['pass_end_location'].apply(lambda v: v[0] if isinstance(v, list) else np.nan)
    events['pass_end_y'] = events['pass_end_location'].apply(lambda v: v[1] if isinstance(v, list) else np.nan)

# taşıma bitiş konumu
if 'carry_end_location' in events.columns:
    events['carry_end_x'] = events['carry_end_location'].apply(lambda v: v[0] if isinstance(v, list) else np.nan)
    events['carry_end_y'] = events['carry_end_location'].apply(lambda v: v[1] if isinstance(v, list) else np.nan)

# Takım isimleri
teams = events['team'].dropna().unique().tolist()
ESP   = next(t for t in teams if 'Spain'   in t)
ENG   = next(t for t in teams if 'England' in t)

print(f'Takımlar: "{ESP}"  vs  "{ENG}"')
print(f'x sütunu boş olmayan event sayısı: {events["x"].notna().sum()}')

Takımlar: "Spain"  vs  "England"
x sütunu boş olmayan event sayısı: 3278


## 4. Maç İstatistikleri

In [8]:
def team_stats(events, team):
    te       = events[events['team'] == team]
    passes   = te[te['type'] == 'Pass']
    shots    = te[te['type'] == 'Shot']
    goals    = shots[shots['shot_outcome'] == 'Goal']
    dribbles = te[te['type'] == 'Dribble']
    presses  = te[te['type'] == 'Pressure']
    return {
        'Pas'           : len(passes),
        'Pas Başarı (%)': round(100 * passes['pass_outcome'].isna().sum() / max(len(passes), 1), 1),
        'Şut'           : len(shots),
        'Gol'           : len(goals),
        'Dribling'      : len(dribbles),
        'Baskı'         : len(presses),
        'Top. Event'    : len(te),
    }

stats = pd.DataFrame([team_stats(events, ESP), team_stats(events, ENG)], index=[ESP, ENG]).T
print('=== MAÇ İSTATİSTİKLERİ ===')
print(stats.to_string())

=== MAÇ İSTATİSTİKLERİ ===
                 Spain  England
Pas              592.0    323.0
Pas Başarı (%)    87.5     78.6
Şut               16.0      9.0
Gol                2.0      1.0
Dribling          11.0     14.0
Baskı            162.0    165.0
Top. Event      2015.0   1289.0


In [9]:
print('=== İSPANYA OYUNCULARI ===')
print(events[events['team']==ESP]['player'].dropna().unique().tolist())
print('\n=== İNGİLTERE OYUNCULARI ===')
print(events[events['team']==ENG]['player'].dropna().unique().tolist())

# Gol atanlar
goals_df = events[(events['type']=='Shot')&(events['shot_outcome']=='Goal')&(events['period']<=4)]
print('\n=== GOLLER ===')
print(goals_df[['player','team','minute','period']].to_string(index=False))

=== İSPANYA OYUNCULARI ===
['Unai Simón Mendibil', 'Robin Aime Robert Le Normand', 'Daniel Carvajal Ramos', 'Álvaro Borja Morata Martín', 'Daniel Olmo Carvajal', 'Rodrigo Hernández Cascante', 'Aymeric Laporte', 'Marc Cucurella Saseta', 'Nicholas Williams Arthuer', 'Fabián Ruiz Peña', 'Lamine Yamal Nasraoui Ebana', 'Martín Zubimendi Ibáñez', 'Mikel Oyarzabal Ugarte', 'José Ignacio Fernández Iglesias', 'Mikel Merino Zazón']

=== İNGİLTERE OYUNCULARI ===
['Kobbie Mainoo', 'Jordan Pickford', 'Jude Bellingham', 'Luke Shaw', 'Declan Rice', 'Marc Guehi', 'Phil Foden', 'Kyle Walker', 'Harry Kane', 'John Stones', 'Bukayo Saka', 'Cole Palmer', 'Ollie Watkins', 'Ivan Toney']

=== GOLLER ===
                   player    team  minute  period
Nicholas Williams Arthuer   Spain      46       2
              Cole Palmer England      72       2
   Mikel Oyarzabal Ugarte   Spain      85       2


In [10]:
# Event sayısına göre en çok katkı veren oyuncular
all_players = events['player'].dropna().unique()
player_counts = events.groupby('player').size().sort_values(ascending=False)
print('=== EN FAZLA EVENT — İSPANYA (TOP 8) ===')
esp_counts = events[events['team']==ESP].groupby('player').size().sort_values(ascending=False)
print(esp_counts.head(8).to_string())
print('\n=== EN FAZLA EVENT — İNGİLTERE (TOP 8) ===')
eng_counts = events[events['team']==ENG].groupby('player').size().sort_values(ascending=False)
print(eng_counts.head(8).to_string())

=== EN FAZLA EVENT — İSPANYA (TOP 8) ===
player
Robin Aime Robert Le Normand    247
Aymeric Laporte                 242
Fabián Ruiz Peña                227
Daniel Carvajal Ramos           216
Nicholas Williams Arthuer       188
Daniel Olmo Carvajal            152
Lamine Yamal Nasraoui Ebana     150
Marc Cucurella Saseta           134

=== EN FAZLA EVENT — İNGİLTERE (TOP 8) ===
player
Jude Bellingham    169
Declan Rice        142
Kyle Walker        135
Bukayo Saka        118
Phil Foden         117
John Stones        111
Jordan Pickford    111
Marc Guehi          90


## 5. İşlenmiş Veriyi Kaydet

In [11]:
meta = {'ESP': ESP, 'ENG': ENG, 'match_id': match_id,
        'competition_id': competition_id, 'season_id': season_id}

with open(os.path.join(CACHE_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False)

pkl_path = os.path.join(CACHE_DIR, 'events_processed.pkl')
events.to_pickle(pkl_path)

print(f'events_processed.pkl kaydedildi  ({os.path.getsize(pkl_path)//1024} KB)')
print('meta.json kaydedildi')
print('\nGörselleştirmeler için 02_visualizations.ipynb dosyasını çalıştırın.')

events_processed.pkl kaydedildi  (2381 KB)
meta.json kaydedildi

Görselleştirmeler için 02_visualizations.ipynb dosyasını çalıştırın.
